# Phase 6 — Per-Subject BAML Prompt Sweep

Interactive walkthrough of the per-subject BAML prompt sweep.
Per the multi-stage plan (see AGENTS.md).

This notebook:
1. Shows the canonical 8-subject → BAML client routing table
2. Sweeps one subject (Mathematics) end-to-end with the stub invoker
3. Demonstrates the `prompt_overlay` future-proofing hook (monkey-patched
   in this Phase 6 baseline; populated from the chunk index later)
4. Aggregates the lift across all 8 subjects

In [ ]:
# 1. Show the 8-subject -> BAML client routing table.
from experiments.prompt_sweeps.prompt_sweep import ALL_8, client_for

print(f"{'subject':20s}  {'baml_client':35s}")
print("-" * 60)
for subject in ALL_8:
    print(f"{subject:20s}  {client_for(subject):35s}")

In [ ]:
# 2. Build a single-subject sweep demo (Mathematics, 3 samples).
import json

from experiments.model_comparison.runner import EvalSample
from experiments.prompt_sweeps.prompt_sweep import sweep_one_subject

MATH_SAMPLES = [
    EvalSample(
        sample_id="math-ie-1",
        pdf_path="ncca.ie/mathematics/en/abc.md",
        subject="mathematics",
        language="en",
        md_text="# Leaving Certificate Mathematics\n## Page 1\n\nAlgebra: linear equations. Calculus: differentiation.\n",
        ground_truth={"subject_slug": "mathematics", "language": "en", "module": "Algebra"},
    ),
    EvalSample(
        sample_id="math-ie-2",
        pdf_path="ncca.ie/mathematics/en/def.md",
        subject="mathematics",
        language="en",
        md_text="# Mathematics\n## Page 1\n\nStatistics: probability. Mechanics: forces.\n",
        ground_truth={"subject_slug": "mathematics", "language": "en", "module": "Statistics"},
    ),
    EvalSample(
        sample_id="math-england-1",
        pdf_path="aqa.org.uk/mathematics/en/ghi.md",
        subject="mathematics",
        language="en",
        md_text="# A-Level Mathematics\n## Page 1\n\nPure Mathematics: algebra, calculus.\n",
        ground_truth={
            "subject_slug": "mathematics",
            "language": "en",
            "module": "Pure Mathematics",
        },
    ),
]


def stub_invoker(client, prompt, overlay, behavior_version):
    # Without overlay: minimal stub (1/3 fields correct)
    # With overlay: perfect response (3/3 fields correct)
    if overlay is not None:
        content = json.dumps({"subject_slug": "mathematics", "language": "en", "module": "Algebra"})
    else:
        content = json.dumps({"subject_slug": "mathematics"})
    return content, 100, 50


result = sweep_one_subject("mathematics", MATH_SAMPLES, behavior_version=1, invoker=stub_invoker)
print(f"Subject: {result.subject}")
print(f"Client:  {result.client}")
print(f"Baseline F1 (no overlay): {result.baseline_f1:.3f}")
print(f"Overlay F1:              {result.overlay_f1:.3f}")
print(f"Lift:                    {result.lift:+.3f}")
print(f"Samples:                 {result.n_samples}")

In [ ]:
# 3. Demonstrate the prompt_overlay hook (Phase 6 baseline returns None).
from experiments.prompt_sweeps.prompt_sweep import build_prompt_overlay

for subject in ALL_8:
    overlay = build_prompt_overlay(subject, behavior_version=42)
    status = "(no overlay yet)" if overlay is None else f"'{overlay[:40]}...'"
    print(f"  {subject:20s}  behavior_version=42  -> overlay={status}")

In [ ]:
# 4. Aggregate lift across all 8 subjects (with stub invoker for the demo).

from experiments.prompt_sweeps.prompt_sweep import sweep_all_subjects


def make_samples(subject):
    return [
        EvalSample(
            sample_id=f"{subject}-1",
            pdf_path="x.pdf",
            subject=subject,
            language="en",
            md_text=f"# {subject}\n## Page 1\n\nstub content",
            ground_truth={"subject_slug": subject, "language": "en"},
        )
    ]


samples_by_subject = {s: make_samples(s) for s in ALL_8}
results = sweep_all_subjects(samples_by_subject, invoker=stub_invoker)

print(f"{'subject':20s}  {'client':35s}  {'base_f1':>8s}  {'overlay_f1':>10s}  {'lift':>8s}")
print("-" * 85)
for r in results:
    print(
        f"{r.subject:20s}  {r.client:35s}  {r.baseline_f1:>8.3f}  {r.overlay_f1:>10.3f}  {r.lift:>+8.3f}"
    )

## Summary

- The 8 `BIEPV3Extract<Subject>` client blocks are wired (Phase 6 commit). Each shares the Vertex AI backend with a per-subject `temperature` (math/science=0.05, humanities=0.1).
- The `prompt_overlay` parameter is added to all 5 LC6 BAML functions + the `prompt_behavior_version` int parameter (for cache invalidation).
- Phase 6 baseline: `build_prompt_overlay()` returns `None` — the canonical default prompts are the baseline. The hook is wired so that data-informed overlays (populated from the chunk index when the PDF pipeline is fully populated) can be added later without changing the BAML function signatures.
- The sweep harness measures lift (F1 with overlay - F1 without). When the overlay is None, lift == 0. When data-informed overlays land (Phase 6 follow-up commit), the sweep will detect positive lifts automatically.
- All 11 sweep tests pass; 8-subject routing table covers the canonical LC roster.
- Ready for Phase 7 (hierarchical visualization) + Phase 8 (local dev + dev deploy).